# AWS Glue 5.0 no Notisbook

Exemplo para o kernel Docker **AWS Glue 5.0**: criamos a sessão com `GlueContext` e gravamos uma tabela Delta externa na camada `bronze`.

In [ ]:
from awsglue.context import GlueContext
from pyspark.context import SparkContext

glueContext = GlueContext(SparkContext.getOrCreate())
spark = glueContext.spark_session

In [ ]:
data = [
    (1, "Ana", "Dados"),
    (2, "Bruno", "Plataforma"),
    (3, "Carla", "Segurança"),
]

df = spark.createDataFrame(data, ["id", "nome", "area"])
echo(df)

Gravamos o DataFrame como tabela Delta **externa** na camada `bronze`: o `path` fica em `s3a://notislake/bronze/` e `overwrite` permite reexecutar a célula.

In [ ]:
(
    df.write
    .format("delta")
    .option("path", "s3a://notislake/bronze/glue_employees_files")
    .mode("overwrite")
    .saveAsTable("bronze.glue_employees")
)

Consultamos a tabela criada acima (célula `spark sql`).

In [ ]:
SELECT * FROM bronze.glue_employees

Também criamos tabelas **Hudi** (Copy-on-Write) na camada `bronze`, uma **gerenciada** e uma **externa**. Toda tabela Hudi exige `primaryKey` e `preCombineField`. Como não confirmamos suporte a `CREATE OR REPLACE` no Hudi, cada exemplo começa com `DROP TABLE IF EXISTS` para poder ser reexecutado.

In [ ]:
spark.sql("DROP TABLE IF EXISTS bronze.glue_hudi_managed")
spark.sql("""
    CREATE TABLE bronze.glue_hudi_managed (id INT, nome STRING, area STRING, ts BIGINT)
    USING hudi
    TBLPROPERTIES (type = 'cow', primaryKey = 'id', preCombineField = 'ts')
""")
spark.sql("""
    INSERT INTO bronze.glue_hudi_managed VALUES
        (1, 'Ana', 'Dados', 1),
        (2, 'Bruno', 'Plataforma', 1),
        (3, 'Carla', 'Segurança', 1)
""")

O `MERGE INTO` atualiza a linha existente (mesmo `id`, `ts` maior) e insere a nova.

In [ ]:
MERGE INTO bronze.glue_hudi_managed AS t
USING (
    SELECT 2 AS id, 'Bruno' AS nome, 'Dados' AS area, CAST(2 AS BIGINT) AS ts
    UNION ALL
    SELECT 4, 'Diego', 'Plataforma', CAST(2 AS BIGINT)
) AS s
ON t.id = s.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

In [ ]:
SELECT id, nome, area FROM bronze.glue_hudi_managed ORDER BY id

A tabela Hudi **externa** aponta para `s3a://notislake/bronze/glue_hudi_externa_files`. Removemos uma linha com `DELETE`; o `DROP TABLE IF EXISTS` mantém os arquivos, e a recriação com o mesmo `LOCATION` reaproveita o que já existe.

In [ ]:
spark.sql("DROP TABLE IF EXISTS bronze.glue_hudi_externa")
spark.sql("""
    CREATE TABLE bronze.glue_hudi_externa (id INT, nome STRING, area STRING, ts BIGINT)
    USING hudi
    TBLPROPERTIES (type = 'cow', primaryKey = 'id', preCombineField = 'ts')
    LOCATION 's3a://notislake/bronze/glue_hudi_externa_files'
""")
spark.sql("""
    INSERT INTO bronze.glue_hudi_externa VALUES
        (1, 'Ana', 'Dados', 1),
        (2, 'Bruno', 'Plataforma', 1),
        (3, 'Carla', 'Segurança', 1)
""")

In [ ]:
DELETE FROM bronze.glue_hudi_externa WHERE id = 3

In [ ]:
SELECT id, nome, area FROM bronze.glue_hudi_externa ORDER BY id